# Notebook 01: Data Preprocessing Fundamentals

#### Overview
This notebook establishes the foundational theory and workflow of Data Preprocessing for Machine Learning. It explores data quality dimensions, data leakage risks, the distinction between raw and clean data, and the architecture of an end-to-end preprocessing pipeline.

# 1. What is Data Preprocessing?

### Concept & Definition
Data Preprocessing is the fundamental stage in an AI/ML pipeline where raw data is cleaned, validated, transformed, and structured into an ML-ready format. Machine Learning algorithms rely strictly on mathematical operations; therefore, unformatted strings, missing entries, scale discrepancies, or noisy records must be resolved prior to model training.

### Real-World / Business Example
A telecom company collects raw customer logs from multiple sources: web sign-up forms, billing databases, and call center logs. The raw data contains missing phone numbers, date strings formatted in different timezones, and text values like `"N/A"` or `"Null"`. Preprocessing converts these fragmented logs into structured numerical vectors suitable for predicting subscriber churn.

### Why It Solves Problems
Raw data cannot be passed directly into scikit-learn models. Preprocessing resolves incompatible data types, fills structural voids, and ensures algorithms receive clean numerical inputs.

In [15]:
import numpy as np
import pandas as pd

# Demonstrating Raw Data vs Clean Structured Data
raw_data = {
    "CustomerID": ["7590-WBEN6", "5575-GNVDE", "3668-QVRZG"],
    "Age": ["34", "NaN", "58"],
    "MonthlyCharges": ["$29.85", "$56.95", "99.99"],
}

df_raw = pd.DataFrame(raw_data)
print("=== Raw Unprocessed DataFrame ===")
print(df_raw)
print("\nRaw Data Types:")
print(df_raw.dtypes)

=== Raw Unprocessed DataFrame ===
   CustomerID  Age MonthlyCharges
0  7590-WBEN6   34         $29.85
1  5575-GNVDE  NaN         $56.95
2  3668-QVRZG   58          99.99

Raw Data Types:
CustomerID        str
Age               str
MonthlyCharges    str
dtype: object


# 2. Why is Data Preprocessing Required?

### Concept & Definition
"Garbage In, Garbage Out" (GIGO) is the primary rule of machine learning. If a model is trained on uncleaned, noisy, or unscaled data, it will learn artificial noise rather than genuine predictive patterns.

### Real-World / Business Example
Consider an algorithm evaluating customer credit risk based on `Annual Salary` (ranging from $20,000 to $500,000) and `Number of Credit Cards` (ranging from 1 to 5). Without scaling, a distance-based model (like KNN) will weigh Salary 100,000 times more heavily than Credit Cards simply due to numerical magnitude.

### Why It Solves Problems
Preprocessing levels the playing field across features, eliminates invalid inputs that cause runtime crashes, and optimizes gradient descent convergence rates during model fitting.

In [16]:
# Demonstrating basic cleaning transformation required before model input
df_cleaned = df_raw.copy()

# Fix string currency issues and handle missing strings
df_cleaned["MonthlyCharges"] = (
    df_cleaned["MonthlyCharges"].str.replace("$", "").astype(float)
)
df_cleaned["Age"] = pd.to_numeric(df_cleaned["Age"], errors="coerce")

print("=== Cleaned DataFrame Ready for Numeric Inspection ===")
print(df_cleaned)
print("\nCleaned Data Types:")
print(df_cleaned.dtypes)

=== Cleaned DataFrame Ready for Numeric Inspection ===
   CustomerID   Age  MonthlyCharges
0  7590-WBEN6  34.0           29.85
1  5575-GNVDE   NaN           56.95
2  3668-QVRZG  58.0           99.99

Cleaned Data Types:
CustomerID            str
Age               float64
MonthlyCharges    float64
dtype: object


# 3. Raw Data vs. Clean Data

### Concept & Definition
- **Raw Data:** Immutable, untouched data extracted directly from primary sources (databases, APIs, logs). It contains noise, inconsistent formatting, incorrect types, missing entries, and duplicates.
- **Clean Data:** Structured, validated, and normalized data produced by applying reproducible data processing workflows.

### Real-World / Business Example
- **Raw Data Entry:** `" 100.50 "` (String with trailing whitespace and currency units).
- **Clean Data Entry:** `100.5` (Pure 64-bit float representation).

### Why It Solves Problems
Machine Learning models require deterministic matrix math operations (dot products, distance metrics). Clean data ensures uniform matrix operations across training batches.

In [17]:
# Inspecting raw file vs cleaned memory state
df_telecom = pd.read_csv("Customer_Data.csv")

print(f"Raw Telecom Dataset Shape: {df_telecom.shape}")
print("\nFirst 3 Raw Records:")
display(df_telecom.head(3))

Raw Telecom Dataset Shape: (1010, 9)

First 3 Raw Records:


,CustomerID,Age,Gender,TenureYears,MonthlyCharges,TotalCharges,ContractType,PaymentMethod,Churn
0,CUST-1000,34.0,Male,3,$29.85,1098.216202,Two year,Credit card,No
1,CUST-1001,150.0,Female,10,$56.95,4544.186090,month to month,Mailed check,Yes
2,CUST-1002,52.0,M,2,$105.50,6144.766598,One year,Credit card,No


# 4. Data Quality

### Concept & Definition
Data Quality measures how effectively a dataset fulfills its intended analytical or machine learning purpose. High-quality data is free from noise, errors, or spurious values that misrepresent reality.

### Real-World / Business Example
In a healthcare predictive system, an recorded body temperature of 105°C (instead of 105°F) represents a severe data quality defect that could corrupt predictive diagnosis models.

### Why It Solves Problems
Evaluating data quality early prevents erroneous model predictions and identifies sensor or data pipeline failures upstream.

In [18]:
# Checking overall data quality metrics programmatically
def check_data_quality(dataframe):
    total_cells = dataframe.size
    null_cells = dataframe.isnull().sum().sum()
    duplicate_rows = dataframe.duplicated().sum()

    quality_score = ((total_cells - null_cells) / total_cells) * 100
    print(f"Overall Data Quality Score: {quality_score:.2f}%")
    print(f"Total Null Values: {null_cells}")
    print(f"Total Duplicate Rows: {duplicate_rows}")


check_data_quality(df_telecom)

Overall Data Quality Score: 98.40%
Total Null Values: 145
Total Duplicate Rows: 10


# 5. Data Consistency

### Concept & Definition
Data Consistency ensures that identical data elements are represented identically across all rows, tables, and system boundaries.

### Real-World / Business Example
If one user enters `"Month-to-month"`, another enters `"month to month"`, and a third enters `"m2m"`, an un-encoded model views these as three completely different categories rather than one uniform plan.

### Why It Solves Problems
Standardizing categories prevents artificial explosion of one-hot encoded column counts and stabilizes group aggregation operations.

In [19]:
# Detecting and fixing categorical inconsistency
inconsistent_series = pd.Series(
    ["Month-to-month", "month to month", "m2m", "One year", "Two year"]
)

print("Before Consistency Standardisation:")
print(inconsistent_series.value_counts())

# Applying standardization mapping
consistency_map = {
    "Month-to-month": "Month-to-month",
    "month to month": "Month-to-month",
    "m2m": "Month-to-month",
    "One year": "One year",
    "Two year": "Two year",
}

consistent_series = inconsistent_series.map(consistency_map)
print("\nAfter Consistency Standardisation:")
print(consistent_series.value_counts())

Before Consistency Standardisation:
Month-to-month    1
month to month    1
m2m               1
One year          1
Two year          1
Name: count, dtype: int64

After Consistency Standardisation:
Month-to-month    3
One year          1
Two year          1
Name: count, dtype: int64


# 6. Data Completeness

### Concept & Definition
Data Completeness indicates the degree to which all required data points are present in the dataset without unexpected missing values or structural gaps.

### Real-World / Business Example
A telecom customer record missing `TotalCharges` or `TenureYears` is structurally incomplete, preventing full lifetime value calculations.

### Why It Solves Problems
Identifying incomplete columns determines whether missing value imputation, row dropping, or synthetic feature generation is necessary.

In [20]:
# Computing completeness percentages per feature
completeness = ((1 - df_telecom.isnull().sum() / len(df_telecom)) * 100).round(
    2
)
completeness_df = pd.DataFrame(
    {"Feature": completeness.index, "Completeness (%)": completeness.values}
)

print("=== Feature Completeness Audit ===")
print(completeness_df)

=== Feature Completeness Audit ===
          Feature  Completeness (%)
0      CustomerID            100.00
1             Age             94.65
2          Gender             95.25
3     TenureYears            100.00
4  MonthlyCharges             95.74
5    TotalCharges            100.00
6    ContractType            100.00
7   PaymentMethod            100.00
8           Churn            100.00


# 7. Data Accuracy

### Concept & Definition
Data Accuracy measures whether recorded values represent the real-world facts they claim to describe.

### Real-World / Business Example
If a customer has a recorded `Age` of 250 years or `MonthlyCharges` of -$50, the data is technically formatted as numbers, but it is factually inaccurate.

### Why It Solves Problems
Filtering inaccurate values prevents impossible outlier points from warping parameter estimates during linear regression or gradient descent steps.

In [21]:
# Checking for Accuracy Violation Anomalies
inaccurate_age = df_telecom[
    (df_telecom["Age"] < 18) | (df_telecom["Age"] > 100)
]
inaccurate_charges = df_telecom[df_telecom["MonthlyCharges"] < 0]

print(f"Inaccurate Age Records Count: {len(inaccurate_age)}")
print(f"Inaccurate Negative Charges Count: {len(inaccurate_charges)}")

TypeError: Invalid comparison between dtype=str and int

# 8. Data Validity

### Concept & Definition
Data Validity measures compliance of data points with predefined schema rules, data types, value ranges, and domain constraints.

### Real-World / Business Example
Validating that `ContractType` contains only accepted categories (`"Month-to-month"`, `"One year"`, `"Two year"`) and rejecting unknown string values.

### Why It Solves Problems
Enforces strict boundary constraints so invalid records are flagged or filtered before entering model execution code.

In [ ]:
# Programmatic Schema & Domain Validity Verification
valid_contracts = {"Month-to-month", "One year", "Two year"}
is_valid_contract = df_telecom["ContractType"].isin(valid_contracts)

print("=== Data Validity Audit ===")
print(f"Valid Contract Records: {is_valid_contract.sum()}")
print(f"Invalid Contract Records: {(~is_valid_contract).sum()}")

# 9. Data Integrity

### Concept & Definition
Data Integrity refers to maintaining the structural relationships, logical dependencies, and referential ties across data features and tables over time.

### Real-World / Business Example
In customer subscription data, `TotalCharges` must logically satisfy the condition $\text{TotalCharges} \approx \text{TenureYears} \times 12 \times \text{MonthlyCharges}$. If `TotalCharges` is non-zero while `TenureYears` is 0, referential data integrity is violated.

### Why It Solves Problems
Prevents conflicting signals where two features contradict each other in the feature matrix.

In [ ]:
# Auditing Logical Integrity Constraints
integrity_violations = df_telecom[
    (df_telecom["TenureYears"] == 0) & (df_telecom["TotalCharges"] > 0)
]

print(f"Integrity Violation Records Count: {len(integrity_violations)}")

# 10. Data Leakage

### Concept & Definition
Data Leakage occurs when information from outside the training dataset (such as target labels or future test distribution statistics) leaks into the training pipeline.

### Real-World / Business Example
Calculating the mean of `MonthlyCharges` using the entire dataset (Train + Test combined) and using it to scale the training set. The model acquires statistical information about the unseen test set, causing high validation scores that fail in deployment.

### Why It Solves Problems
Preventing data leakage guarantees that your model performance metrics realistically reflect production performance.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# WRONG WORKFLOW (Causes Data Leakage): Scaling before splitting
scaler_wrong = StandardScaler()
df_scaled_wrong = scaler_wrong.fit_transform(df_telecom[["MonthlyCharges"]])
X_tr_w, X_te_w = train_test_split(
    df_scaled_wrong, test_size=0.2, random_state=42
)

# CORRECT WORKFLOW (Prevents Data Leakage): Split FIRST, then fit scaler ONLY on Train
X_train_raw, X_test_raw = train_test_split(
    df_telecom[["MonthlyCharges"]], test_size=0.2, random_state=42
)

scaler_correct = StandardScaler()
X_train_clean = scaler_correct.fit_transform(X_train_raw)  # fit ONLY on train
X_test_clean = scaler_correct.transform(X_test_raw)  # transform test using train fit

print("Data Leakage Prevention Verification Completed Cleanly.")

# 11. Training Data

### Concept & Definition
Training Data is the core subset of the dataset (typically 70%–80%) used exclusively to fit algorithm parameters and compute preprocessing parameters (imputer medians, scale means, encoder levels).

### Real-World / Business Example
Fitting a decision tree or neural network on 5,634 telecom records so it learns patterns associated with customer churn.

### Why It Solves Problems
Provides the foundation for pattern learning without exposing the model to evaluation benchmarks.

In [ ]:
print(f"Training Set Rows: {len(X_train_raw)}")
print(f"Training Set Percentage: {(len(X_train_raw)/len(df_telecom))*100:.1f}%")

# 12. Validation Data

### Concept & Definition
Validation Data is an independent sample partition used during model development to tune hyperparameters (e.g., tree depth, regularization strength) and choose the best model architecture.

### Real-World / Business Example
Comparing a Random Forest vs. XGBoost model on the validation set to decide which architecture to deploy.

### Why It Solves Problems
Prevents overfitting to the training set by providing an intermediate evaluation step before final testing.

In [ ]:
# Creating Train, Validation, and Test Partition Splits (70% Train, 15% Val, 15% Test)
X_train_sub, X_val, y_train_sub, y_val = train_test_split(
    X_train_raw,
    df_telecom.loc[X_train_raw.index, "Churn"],
    test_size=0.1875,
    random_state=42,
)

print(f"Sub-Train Set Count: {len(X_train_sub)}")
print(f"Validation Set Count: {len(X_val)}")

# 13. Test Data

### Concept & Definition
Test Data is a completely isolated holdout dataset used exclusively at the very end of the pipeline to measure final unbiased model generalization.

### Real-World / Business Example
Evaluating final model churn precision on unseen customer accounts to estimate actual dollar savings before deployment.

### Why It Solves Problems
Serves as an objective benchmark of production readiness.

In [ ]:
print(f"Final Isolated Test Set Count: {len(X_test_raw)}")
print(f"Test Set Percentage: {(len(X_test_raw)/len(df_telecom))*100:.1f}%")

# 14. Preprocessing Pipeline

### Concept & Definition
A Preprocessing Pipeline wraps data cleaning, imputation, encoding, and scaling steps into a single object. It executes all data transformations sequentially and guarantees zero data leakage between splits.

### Real-World / Business Example
Deploying an end-to-end `ColumnTransformer` + `Pipeline` architecture to web services so live production inputs are transformed automatically using training-set parameters.

### Why It Solves Problems
Eliminates manual reproduction of cleaning steps and guarantees consistent transformations between training and inference.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Separate features and target
X = df_telecom.drop(columns=["CustomerID", "Churn"])
y = df_telecom["Churn"]

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Identify numerical and categorical columns
num_features = X_train.select_dtypes(include=["number"]).columns
cat_features = X_train.select_dtypes(include=["object"]).columns

# Build Sub-Pipelines
num_pipe = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

cat_pipe = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]
)

# Complete Preprocessing Pipeline
full_pipeline = ColumnTransformer(
    transformers=[
        ("num", num_pipe, num_features),
        ("cat", cat_pipe, cat_features),
    ]
)

# Fit on Train, Transform on Train & Test
X_train_processed = full_pipeline.fit_transform(X_train)
X_test_processed = full_pipeline.transform(X_test)

print("=== Preprocessing Pipeline Execution Complete ===")
print(f"Processed Training Shape: {X_train_processed.shape}")
print(f"Processed Test Shape: {X_test_processed.shape}")